# Notebook 02 — Condition A: XGBoost with Median Imputation

## Requires
- `data/raw/`

## Produces
- `results/models/xgb_condition_A.pkl`
- Row appended to `results/experiment_log.csv`

## Feature count
48 lag features added after the patient-level split (8 vitals × lag1/lag2/lag4/roll4_mean/roll4_std/delta1). Combined with 40 original features: **88 total features**.

In [ ]:
import sys
sys.path.insert(0, '..')

import os
import numpy as np
import joblib
import xgboost as xgb
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score

from src.config import RANDOM_SEED, DATA_DIR, MODELS_DIR
from src.data_loader import load_all_patients
from src.preprocessing import engineer_labels, clip_outliers
from src.features import add_lag_features
from src.utils import set_all_seeds, validate_no_nans, create_patient_splits
from src.evaluate import select_threshold, compute_all_metrics, log_results

set_all_seeds(RANDOM_SEED)
print('Imports OK')

## 1. Load Data and Create Splits

Loading the full dataset and applying the stratified 70/15/15 patient-level split. Stratification preserves the 5.62% sepsis prevalence across train, validation, and test sets.

In [ ]:
df = load_all_patients(DATA_DIR)
df, _ = engineer_labels(df)
df = clip_outliers(df)

patient_train, patient_val, patient_test = create_patient_splits(df)

train_df = df[df['patient_id'].isin(patient_train)].copy()
val_df   = df[df['patient_id'].isin(patient_val)].copy()
test_df  = df[df['patient_id'].isin(patient_test)].copy()

print(f'Train: {len(patient_train):,} | Val: {len(patient_val):,} | Test: {len(patient_test):,}')

## 2. Lag Feature Engineering

Adding 48 temporal lag features across the 8 vital signs — computed after the split to prevent data leakage. These give XGBoost explicit access to trend information (rate of change, rolling averages) that it could not otherwise derive from a single-timestep snapshot.

In [ ]:
# Per-patient groupby — no cross-patient leakage
train_df = add_lag_features(train_df)
val_df   = add_lag_features(val_df)
test_df  = add_lag_features(test_df)
print(f'Columns after lag features: {len(train_df.columns)}')

## 3. Strategy A Preprocessing

Applying median imputation using per-feature training medians, followed by standard scaling. Imputation parameters are fitted on the training set only and applied to validation and test sets.

In [ ]:
m
e
t
a
_
c
o
l
s
 
 
 
 
=
 
{
'
p
a
t
i
e
n
t
_
i
d
'
,
 
'
h
o
s
p
i
t
a
l
_
i
d
'
,
 
'
S
e
p
s
i
s
L
a
b
e
l
'
,
 
'
E
a
r
l
y
L
a
b
e
l
'
}

f
e
a
t
u
r
e
_
c
o
l
s
 
=
 
[
c
 
f
o
r
 
c
 
i
n
 
t
r
a
i
n
_
d
f
.
c
o
l
u
m
n
s
 
i
f
 
c
 
n
o
t
 
i
n
 
m
e
t
a
_
c
o
l
s
]


X
_
t
r
a
i
n
_
r
a
w
 
=
 
t
r
a
i
n
_
d
f
[
f
e
a
t
u
r
e
_
c
o
l
s
]
.
v
a
l
u
e
s

X
_
v
a
l
_
r
a
w
 
 
 
=
 
v
a
l
_
d
f
[
f
e
a
t
u
r
e
_
c
o
l
s
]
.
v
a
l
u
e
s

X
_
t
e
s
t
_
r
a
w
 
 
=
 
t
e
s
t
_
d
f
[
f
e
a
t
u
r
e
_
c
o
l
s
]
.
v
a
l
u
e
s

y
_
t
r
a
i
n
 
=
 
t
r
a
i
n
_
d
f
[
'
E
a
r
l
y
L
a
b
e
l
'
]
.
v
a
l
u
e
s

y
_
v
a
l
 
 
 
=
 
v
a
l
_
d
f
[
'
E
a
r
l
y
L
a
b
e
l
'
]
.
v
a
l
u
e
s

y
_
t
e
s
t
 
 
=
 
t
e
s
t
_
d
f
[
'
E
a
r
l
y
L
a
b
e
l
'
]
.
v
a
l
u
e
s


i
m
p
u
t
e
r
 
=
 
S
i
m
p
l
e
I
m
p
u
t
e
r
(
s
t
r
a
t
e
g
y
=
'
m
e
d
i
a
n
'
)

X
_
t
r
a
i
n
 
=
 
i
m
p
u
t
e
r
.
f
i
t
_
t
r
a
n
s
f
o
r
m
(
X
_
t
r
a
i
n
_
r
a
w
)

X
_
v
a
l
 
 
 
=
 
i
m
p
u
t
e
r
.
t
r
a
n
s
f
o
r
m
(
X
_
v
a
l
_
r
a
w
)

X
_
t
e
s
t
 
 
=
 
i
m
p
u
t
e
r
.
t
r
a
n
s
f
o
r
m
(
X
_
t
e
s
t
_
r
a
w
)


s
c
a
l
e
r
 
 
=
 
S
t
a
n
d
a
r
d
S
c
a
l
e
r
(
)

X
_
t
r
a
i
n
 
=
 
s
c
a
l
e
r
.
f
i
t
_
t
r
a
n
s
f
o
r
m
(
X
_
t
r
a
i
n
)

X
_
v
a
l
 
 
 
=
 
s
c
a
l
e
r
.
t
r
a
n
s
f
o
r
m
(
X
_
v
a
l
)

X
_
t
e
s
t
 
 
=
 
s
c
a
l
e
r
.
t
r
a
n
s
f
o
r
m
(
X
_
t
e
s
t
)


f
o
r
 
a
r
r
,
 
n
a
m
e
 
i
n
 
[
(
X
_
t
r
a
i
n
,
'
t
r
a
i
n
_
A
'
)
,
(
X
_
v
a
l
,
'
v
a
l
_
A
'
)
,
(
X
_
t
e
s
t
,
'
t
e
s
t
_
A
'
)
]
:

 
 
 
 
v
a
l
i
d
a
t
e
_
n
o
_
n
a
n
s
(
a
r
r
,
 
n
a
m
e
,
 
f
e
a
t
u
r
e
_
c
o
l
s
)


o
s
.
m
a
k
e
d
i
r
s
(
M
O
D
E
L
S
_
D
I
R
,
 
e
x
i
s
t
_
o
k
=
T
r
u
e
)

j
o
b
l
i
b
.
d
u
m
p
(
i
m
p
u
t
e
r
,
 
 
 
 
 
 
f
'
{
M
O
D
E
L
S
_
D
I
R
}
s
t
r
a
t
e
g
y
_
A
_
l
a
g
_
i
m
p
u
t
e
r
.
p
k
l
'
)

j
o
b
l
i
b
.
d
u
m
p
(
s
c
a
l
e
r
,
 
 
 
 
 
 
 
f
'
{
M
O
D
E
L
S
_
D
I
R
}
s
t
r
a
t
e
g
y
_
A
_
l
a
g
_
s
c
a
l
e
r
.
p
k
l
'
)

j
o
b
l
i
b
.
d
u
m
p
(
f
e
a
t
u
r
e
_
c
o
l
s
,
 
f
'
{
M
O
D
E
L
S
_
D
I
R
}
s
t
r
a
t
e
g
y
_
A
_
l
a
g
_
f
e
a
t
u
r
e
_
n
a
m
e
s
.
p
k
l
'
)

p
r
i
n
t
(
f
'
F
e
a
t
u
r
e
s
:
 
{
X
_
t
r
a
i
n
.
s
h
a
p
e
[
1
]
}
 
|
 
T
r
a
i
n
 
r
o
w
s
:
 
{
X
_
t
r
a
i
n
.
s
h
a
p
e
[
0
]
:
,
}
 
|
 
m
e
a
n
 
≈
 
{
X
_
t
r
a
i
n
.
m
e
a
n
(
)
:
.
4
f
}
'
)

## 4. XGBoost Hyperparameter Search

Grid search over tree depth ∈ {3, 4, 6} and learning rate ∈ {0.05, 0.1}, with 500 estimators and early stopping on validation AUPRC (patience = 20). Class imbalance is handled via `scale_pos_weight = 43.3`.

In [ ]:
pos_weight = (len(y_train) - y_train.sum()) / y_train.sum()
print(f'scale_pos_weight = {pos_weight:.1f}')

param_grid = [{'max_depth': d, 'learning_rate': lr}
              for d in [3, 4, 6] for lr in [0.05, 0.1]]

best_val_auprc, best_params, best_model = 0.0, None, None

for params in param_grid:
    m = xgb.XGBClassifier(
        n_estimators=500, max_depth=params['max_depth'],
        learning_rate=params['learning_rate'],
        scale_pos_weight=pos_weight, eval_metric='aucpr',
        early_stopping_rounds=20, random_state=RANDOM_SEED, n_jobs=-1,
    )
    m.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    val_auprc = average_precision_score(y_val, m.predict_proba(X_val)[:, 1])
    print(f"depth={params['max_depth']} lr={params['learning_rate']} val AUPRC={val_auprc:.4f}")
    if val_auprc > best_val_auprc:
        best_val_auprc, best_params, best_model = val_auprc, params, m

print(f'Best: {best_params}  val AUPRC: {best_val_auprc:.4f}')

## 5. Evaluate and Log

Evaluating the best model on the held-out test set and appending results to `experiment_log.csv`. Reporting AUPRC, AUC-ROC, F1, precision, and recall at the threshold that maximised F1 on the validation set.

In [ ]:
val_probs  = best_model.predict_proba(X_val)[:, 1]
test_probs = best_model.predict_proba(X_test)[:, 1]

threshold    = select_threshold(y_val, val_probs)
val_metrics  = compute_all_metrics(y_val,  val_probs,  threshold)
test_metrics = compute_all_metrics(y_test, test_probs, threshold)

print(f'Test AUC-ROC: {test_metrics["auc_roc"]:.4f} | AUPRC: {test_metrics["auprc"]:.4f} | F1: {test_metrics["f1"]:.4f}')

log_results('A', 'XGBoost', 'Strategy_A', val_metrics, test_metrics, best_params)

joblib.dump(best_model, f'{MODELS_DIR}xgb_condition_A.pkl')
print(f'Saved | Features: {best_model.n_features_in_}')